In [1]:
import numpy as np
import pandas as pd
import ast
from collections import defaultdict
import pymorphy3
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import plotly.express as px

C:\Users\topi7\OneDrive\Desktop\FEFU\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("povarenok.csv")


In [3]:
df = df.drop(['url'], axis=1)

In [4]:
df

,name,ingredients
0,Густой молочно-клубничный коктейль,"{'Молоко': '250 мл', 'Клубника': '200 г', 'Сах..."
1,Рулетики,"{'Сыр твердый': None, 'Чеснок': None, 'Яйцо ку..."
2,"Салат ""Очищение души и кишечника""","{'Баклажан': '3 шт', 'Лук репчатый': '2 шт', '..."
3,Куриные котлеты с картофельным пюре в духовке,"{'Фарш куриный': '800 г', 'Пюре картофельное':..."
4,Рецепт вишневой наливки,"{'Вишня': '1 кг', 'Водка': '1 л', 'Сахар': '30..."
...,...,...
146577,"Украшение для блюд ""Снежинка""","{'Капуста краснокочанная': None, 'Капуста бело..."
146578,Греческий рисовый пирог с фаршем,"{'Фарш мясной': '400 г', 'Масло сливочное': '2..."
146579,Соус на груздях с хреном,"{'Грибы': '300 г', 'Чеснок': '5 зуб.', 'Лук ре..."
146580,Печенье со сливочным сыром,"{'Масло сливочное': '225 г', 'Сыр сливочный': ..."


In [5]:
df['name'].notna().any()

True

In [6]:
df['ingredients'].notna().any()


True

In [7]:
# df = df[:10].copy()
df = df[:5800]

In [8]:
df['ingredients_dict'] = df['ingredients'].apply(ast.literal_eval)

C:\Users\topi7\AppData\Local\Temp\ipykernel_5956\1089393177.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ingredients_dict'] = df['ingredients'].apply(ast.literal_eval)


In [9]:
df = df.drop(['ingredients'], axis=1)

In [10]:
df['ingredients_dict'] = df['ingredients_dict'].apply(lambda x: x.keys())

In [11]:
df

,name,ingredients_dict
0,Густой молочно-клубничный коктейль,"(Молоко, Клубника, Сахар)"
1,Рулетики,"(Сыр твердый, Чеснок, Яйцо куриное, Грейпфрут,..."
2,"Салат ""Очищение души и кишечника""","(Баклажан, Лук репчатый, Помидор, Чеснок, Соль..."
3,Куриные котлеты с картофельным пюре в духовке,"(Фарш куриный, Пюре картофельное, Желток яичны..."
4,Рецепт вишневой наливки,"(Вишня, Водка, Сахар, Кости, Корица, Гвоздика)"
...,...,...
5795,"Салат ""Для любимого мужчины""","(Горбуша, Картофель, Редис, Листья салата, Лук..."
5796,"Кускус ""На завтрак"" с орехами и сухофруктами","(Кус-кус, Курага, Чернослив, Орехи кешью, Орех..."
5797,"Блины ""Праздничные""","(Молоко, Яйцо куриное, Мука пшеничная, Масло п..."
5798,Кесо Фреско,"(Молоко, Закваска, Хлорид кальция, Сычужный фе..."


In [12]:
df['ingredients_dict'].notna().any()

True

In [13]:
df['ingredients_dict'] = df['ingredients_dict'].apply(lambda x: sorted(list(x)))


In [14]:
df['ingredients_list'] = df['ingredients_dict'].apply(
    lambda lst: [item.lower() for item in lst]
)

In [15]:
df = df.drop(['ingredients_dict'], axis=1)
df

,name,ingredients_list
0,Густой молочно-клубничный коктейль,"[клубника, молоко, сахар]"
1,Рулетики,"[грейпфрут, листья салата, лук зеленый, майоне..."
2,"Салат ""Очищение души и кишечника""","[баклажан, лук репчатый, майонез, перец черный..."
3,Куриные котлеты с картофельным пюре в духовке,"[желток яичный, зелень, лук репчатый, масло ра..."
4,Рецепт вишневой наливки,"[вишня, водка, гвоздика, корица, кости, сахар]"
...,...,...
5795,"Салат ""Для любимого мужчины""","[горбуша, картофель, листья салата, лук зелены..."
5796,"Кускус ""На завтрак"" с орехами и сухофруктами","[курага, кус-кус, масло сливочное, орехи грецк..."
5797,"Блины ""Праздничные""","[масло подсолнечное, масло сливочное, молоко, ..."
5798,Кесо Фреско,"[закваска, молоко, соль, сычужный фермент, хло..."


## Лемматизация

In [16]:
morph = pymorphy3.MorphAnalyzer()

In [17]:
a = morph.parse('стали')[0]
a.normal_form

'стать'

In [18]:
df['ingredients_lemmatized'] = df['ingredients_list'].apply(
    lambda x: [morph.parse(i)[0].normal_form for i in x]
)

In [19]:
df

,name,ingredients_list,ingredients_lemmatized
0,Густой молочно-клубничный коктейль,"[клубника, молоко, сахар]","[клубника, молоко, сахар]"
1,Рулетики,"[грейпфрут, листья салата, лук зеленый, майоне...","[грейпфрут, листья салат, лук зелёный, майонез..."
2,"Салат ""Очищение души и кишечника""","[баклажан, лук репчатый, майонез, перец черный...","[баклажан, лук репчатый, майонез, перец чёрный..."
3,Куриные котлеты с картофельным пюре в духовке,"[желток яичный, зелень, лук репчатый, масло ра...","[желток яичный, зелень, лук репчатый, масло ра..."
4,Рецепт вишневой наливки,"[вишня, водка, гвоздика, корица, кости, сахар]","[вишня, водка, гвоздик, корица, кость, сахар]"
...,...,...,...
5795,"Салат ""Для любимого мужчины""","[горбуша, картофель, листья салата, лук зелены...","[горбуша, картофель, листья салат, лук зелёный..."
5796,"Кускус ""На завтрак"" с орехами и сухофруктами","[курага, кус-кус, масло сливочное, орехи грецк...","[курага, кус-кус, масло сливочный, орехи грецк..."
5797,"Блины ""Праздничные""","[масло подсолнечное, масло сливочное, молоко, ...","[масло подсолнечный, масло сливочный, молоко, ..."
5798,Кесо Фреско,"[закваска, молоко, соль, сычужный фермент, хло...","[закваска, молоко, соль, сычужный фермент, хло..."


## Частотный анализ

In [20]:

all_ingredients = df['ingredients_lemmatized'].sum()

In [21]:
values, counts = np.unique(all_ingredients, return_counts=True)

In [22]:
frequency_analysis = dict(zip(values,counts ))

In [24]:
frequency_analysis = dict(sorted(frequency_analysis.items(), key=lambda item: item[1], reverse=True))


In [25]:
frequency_analysis

{'соль': 3125,
 'яйцо куриный': 2342,
 'мука пшеничный': 2046,
 'сахар': 1966,
 'лук репчатый': 1656,
 'масло растительный': 1573,
 'масло сливочный': 1526,
 'чеснок': 1229,
 'перец чёрный': 1131,
 'вода': 1019,
 'молоко': 955,
 'морковь': 898,
 'сметана': 750,
 'майонез': 690,
 'картофель': 647,
 'разрыхлитель тест': 631,
 'сыр твёрдый': 611,
 'помидор': 604,
 'масло оливковый': 532,
 'зелень': 530,
 'сливка': 490,
 'специя': 478,
 'соевый соусый': 468,
 'перец болгарский': 419,
 'дрожжи': 385,
 'сахарная пудрый': 371,
 'петрушка': 369,
 'сок лимонный': 363,
 'творог': 356,
 'мёд': 350,
 'уксус': 345,
 'укроп': 335,
 'яблоко': 332,
 'сода': 316,
 'рис': 307,
 'ванильный сахар': 305,
 'огурец': 295,
 'ванилин': 294,
 'корица': 283,
 'орехи грецкий': 276,
 'желток яичный': 272,
 'лимон': 271,
 'какао-порошок': 270,
 'лук зелёный': 256,
 'шампиньон': 240,
 'приправа': 237,
 'горчица': 234,
 'фарш мясной': 228,
 'масло подсолнечный': 225,
 'лист лавровый': 215,
 'гриб': 207,
 'томатная па

## BOW

In [26]:
# unique_ingredients = sorted(set(all_ingredients))
# df['BOW'] = df['ingredients_lemmatized'].apply(lambda x: [int(i in x) for i in unique_ingredients])
# df['BOW']

In [27]:
exploded = df.explode('ingredients_lemmatized')

In [28]:
dummies  = pd.get_dummies(exploded['ingredients_lemmatized'])

In [29]:
bow = dummies.groupby(exploded.index).max()


In [30]:
bow

,абрикос,авокадо,агар-агар,аджика,айва,алкоголь,алыча,аммоний пищев,ананас,анис,...,экстракт,эссенция,эстрагон,яблоко,ягода,язык говяжий,язык свиной,яйцо куриный,яйцо перепелиный,ёрш
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5795,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5796,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5797,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
5798,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## BPE


In [31]:
corpus = []

for idx, ingredient in enumerate(frequency_analysis):
    a = []
    for i in ingredient:
        if i ==' ':
            a.append('</w>')
        else:
            a.append(i)
    a.append('</w>')
    corpus.append(a)
    corpus[idx].append(int(frequency_analysis[ingredient]))

In [32]:
corpus

[['с', 'о', 'л', 'ь', '</w>', 3125],
 ['я', 'й', 'ц', 'о', '</w>', 'к', 'у', 'р', 'и', 'н', 'ы', 'й', '</w>', 2342],
 ['м',
  'у',
  'к',
  'а',
  '</w>',
  'п',
  'ш',
  'е',
  'н',
  'и',
  'ч',
  'н',
  'ы',
  'й',
  '</w>',
  2046],
 ['с', 'а', 'х', 'а', 'р', '</w>', 1966],
 ['л', 'у', 'к', '</w>', 'р', 'е', 'п', 'ч', 'а', 'т', 'ы', 'й', '</w>', 1656],
 ['м',
  'а',
  'с',
  'л',
  'о',
  '</w>',
  'р',
  'а',
  'с',
  'т',
  'и',
  'т',
  'е',
  'л',
  'ь',
  'н',
  'ы',
  'й',
  '</w>',
  1573],
 ['м',
  'а',
  'с',
  'л',
  'о',
  '</w>',
  'с',
  'л',
  'и',
  'в',
  'о',
  'ч',
  'н',
  'ы',
  'й',
  '</w>',
  1526],
 ['ч', 'е', 'с', 'н', 'о', 'к', '</w>', 1229],
 ['п', 'е', 'р', 'е', 'ц', '</w>', 'ч', 'ё', 'р', 'н', 'ы', 'й', '</w>', 1131],
 ['в', 'о', 'д', 'а', '</w>', 1019],
 ['м', 'о', 'л', 'о', 'к', 'о', '</w>', 955],
 ['м', 'о', 'р', 'к', 'о', 'в', 'ь', '</w>', 898],
 ['с', 'м', 'е', 'т', 'а', 'н', 'а', '</w>', 750],
 ['м', 'а', 'й', 'о', 'н', 'е', 'з', '</w>', 690],
 ['

In [33]:
def get_pair_frequencies(corpus):
    pair_freq = defaultdict(int)

    for word in corpus:

        freq = word[-1]
        symbols = word[:-1]


        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pair_freq[pair] += freq

    return pair_freq

In [34]:
pair_freq = get_pair_frequencies(corpus)
best_pair = max(pair_freq, key=pair_freq.get)
print(best_pair, pair_freq[best_pair])

('й', '</w>') 21314


In [35]:
def merge_pair(pair, corpus):
    new_corpus = []

    a, b = pair
    merged = a + b

    for word in corpus:
        freq = word[-1]
        symbols = word[:-1]

        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        new_corpus.append(new_symbols + [freq])

    return new_corpus


In [36]:
merge_pair(best_pair, corpus)

[['с', 'о', 'л', 'ь', '</w>', 3125],
 ['я', 'й', 'ц', 'о', '</w>', 'к', 'у', 'р', 'и', 'н', 'ы', 'й</w>', 2342],
 ['м',
  'у',
  'к',
  'а',
  '</w>',
  'п',
  'ш',
  'е',
  'н',
  'и',
  'ч',
  'н',
  'ы',
  'й</w>',
  2046],
 ['с', 'а', 'х', 'а', 'р', '</w>', 1966],
 ['л', 'у', 'к', '</w>', 'р', 'е', 'п', 'ч', 'а', 'т', 'ы', 'й</w>', 1656],
 ['м',
  'а',
  'с',
  'л',
  'о',
  '</w>',
  'р',
  'а',
  'с',
  'т',
  'и',
  'т',
  'е',
  'л',
  'ь',
  'н',
  'ы',
  'й</w>',
  1573],
 ['м',
  'а',
  'с',
  'л',
  'о',
  '</w>',
  'с',
  'л',
  'и',
  'в',
  'о',
  'ч',
  'н',
  'ы',
  'й</w>',
  1526],
 ['ч', 'е', 'с', 'н', 'о', 'к', '</w>', 1229],
 ['п', 'е', 'р', 'е', 'ц', '</w>', 'ч', 'ё', 'р', 'н', 'ы', 'й</w>', 1131],
 ['в', 'о', 'д', 'а', '</w>', 1019],
 ['м', 'о', 'л', 'о', 'к', 'о', '</w>', 955],
 ['м', 'о', 'р', 'к', 'о', 'в', 'ь', '</w>', 898],
 ['с', 'м', 'е', 'т', 'а', 'н', 'а', '</w>', 750],
 ['м', 'а', 'й', 'о', 'н', 'е', 'з', '</w>', 690],
 ['к', 'а', 'р', 'т', 'о', 'ф', '

In [37]:
def bpe(n, corpus):
    for j in range(n):
        pair_freq = get_pair_frequencies(corpus)
        best_pair = max(pair_freq, key=pair_freq.get)

        corpus = merge_pair(best_pair, corpus)
    return corpus

In [38]:

bpe(22, corpus)


[['со', 'ль</w>', 3125],
 ['я', 'й', 'ц', 'о</w>', 'ку', 'ри', 'ный</w>', 2342],
 ['м', 'ук', 'а</w>', 'п', 'ш', 'е', 'ни', 'чный</w>', 2046],
 ['с', 'а', 'х', 'ар', '</w>', 1966],
 ['л', 'ук', '</w>', 'ре', 'п', 'ч', 'а', 'т', 'ый</w>', 1656],
 ['масло</w>', 'р', 'ас', 'т', 'и', 'т', 'е', 'ль', 'ный</w>', 1573],
 ['масло</w>', 'с', 'ли', 'в', 'о', 'чный</w>', 1526],
 ['ч', 'е', 'с', 'н', 'ок', '</w>', 1229],
 ['пе', 'ре', 'ц', '</w>', 'ч', 'ё', 'р', 'ный</w>', 1131],
 ['в', 'о', 'д', 'а</w>', 1019],
 ['м', 'о', 'л', 'ок', 'о</w>', 955],
 ['м', 'о', 'р', 'к', 'о', 'в', 'ь', '</w>', 898],
 ['с', 'м', 'е', 'т', 'а', 'н', 'а</w>', 750],
 ['м', 'а', 'й', 'о', 'н', 'е', 'з', '</w>', 690],
 ['к', 'ар', 'т', 'о', 'ф', 'е', 'ль</w>', 647],
 ['р',
  'а',
  'з',
  'р',
  'ы',
  'х',
  'ли',
  'т',
  'е',
  'ль</w>',
  'т',
  'е',
  'с',
  'т',
  '</w>',
  631],
 ['с', 'ы', 'р', '</w>', 'т', 'в', 'ё', 'р', 'д', 'ый</w>', 611],
 ['п', 'о', 'м', 'и', 'д', 'о', 'р', '</w>', 604],
 ['масло</w>', 'о',

* Чем больше n
  - меньше токенов в тексте
  - чаще появляются целые слова
  - словарь раздувается
  - хуже обобщение на новые слова
* Чем меньше n
  - маленький словарь
  - лучше морфологическое обобщение
  - текст длиннее
  - больше шагов для модели


## BertTokenizer

In [39]:
tokenizer = BertTokenizer.from_pretrained(
    "bert-base-multilingual-cased"
)

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-multilingual-cased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 47b58cad-51d8-424a-a7b2-31e70486aa49)')' thrown while requesting HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-multilingual-cased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 769c07d8-e4a8-4c67-b90a-2e0b99f2e69b)')' thrown while requesting HEAD https://huggingface.co/bert-base-multilingual-cased/res

In [40]:
text = "масло сливочное"

tokens = tokenizer.tokenize(text)
print(tokens)

['ма', '##сло', 'сл', '##иво', '##чное']


In [41]:
encoded = tokenizer.encode(
    text,
    add_special_tokens=True
)


print(encoded)

[101, 97744, 55984, 52399, 72352, 67482, 102]


In [42]:
model = BertModel.from_pretrained("bert-base-multilingual-cased")

In [43]:
ingredients = ["клубника", "молоко", "сахар"]
text = ", ".join(ingredients)

inputs = tokenizer(text, return_tensors="pt")

In [44]:
inputs

{'input_ids': tensor([[   101,  16233,  14071,    117,    553,  69605,  11623,    117,  10868,
         104888,    102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

outputs.last_hidden_state → [batch_size, seq_len, hidden_size]

In [45]:
with torch.no_grad(): #не хранить градиенты
    outputs = model(**inputs)

embedding = outputs.last_hidden_state[:, 0, :]

In [46]:
embedding

tensor([[ 9.3631e-02, -1.0672e-01,  3.0365e-01,  2.1267e-01,  3.7680e-01,
          3.8497e-02,  1.2048e-01, -3.0851e-02,  8.0611e-02,  2.8533e-01,
          5.0185e-02,  1.5418e-01,  5.9505e-02,  3.0240e-01, -7.2601e-01,
         -2.8574e-01, -1.6038e-01,  3.2285e-01, -1.3725e-01,  2.0230e-01,
          2.2320e-01,  1.4192e-02, -5.1614e-01, -9.0738e-02,  1.0570e-01,
         -4.3999e-02, -4.3023e-01, -2.0169e-01,  2.4216e-01,  1.7035e-01,
          1.8687e-01, -1.2239e-01,  2.0965e-02, -1.8346e-02,  1.6845e-01,
         -6.7431e-02, -1.6433e+00, -3.1107e-01,  9.9729e-02, -8.2037e-02,
         -3.3351e-01,  2.5322e-01, -1.1198e-03,  4.1166e-02, -5.6697e-02,
          1.2303e+00, -1.0506e-01,  2.0632e-02,  1.4414e+00, -9.9963e-01,
          2.4011e-01, -6.3293e-01,  9.3214e-02, -1.6326e+00,  7.9976e-03,
          2.3857e-01, -4.2891e-02, -3.1507e-02,  3.1134e-02,  1.7240e-01,
         -9.8224e-02, -7.9554e-02,  2.5416e-01,  2.6955e-01, -3.1363e-01,
         -3.4742e-02,  4.5743e-02,  1.

In [47]:
embeddings = []
for ingredients  in df['ingredients_list']:
    text = ", ".join(ingredients)
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad(): #не хранить градиенты
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state[:, 0, :][0]
    embeddings.append(embedding.numpy())
df['embeddings'] = embeddings

Косинусное сходство нового рецепта с каждым рецептом базы вычисляется так:

$$
\text{similarities}[i] = \frac{a \cdot b_i}{\|a\| \, \|b_i\|}, \quad \forall i = 1, \dots, N
$$



1 → векторы максимально похожи

0 → нейтрально, не похожи

-1 → противоположны

## WORD2VEC

In [51]:
recipes = df['ingredients_list'].tolist()

In [52]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=recipes,
    vector_size=100,   # сколько признаков описывает слово
    window=5,          # сколько слов считается контекстом
    min_count=2,       # минимальное число вхождений слова
    sg=1,              # Skip-gram - угадывает контекст по слову
    workers=4,         # число потоков CPU
    epochs=20
)


In [53]:
model.wv # - эмбединги

In [118]:
salad_vec = model.wv['салат']

KeyError: "Key 'салат' not present"

In [55]:
recipe_names = df['name'].to_list()


In [56]:
def recipe_embedding(recipe, model):
    vecs = [model.wv[w] for w in recipe if w in model.wv]
    return np.mean(vecs, axis=0)

In [57]:
recipe_vectors = np.array([recipe_embedding(r, model) for r in recipes])


In [58]:

pca = PCA(n_components=3)
reduced = pca.fit_transform(recipe_vectors)

fig = px.scatter_3d(
    x=reduced[:,0],
    y=reduced[:,1],
    z=reduced[:,2],
    hover_name=recipe_names,
    size_max=5,
    title="3D визуализация пространства рецептов"
)
fig.show()


Выглядит похожим на правду.

In [59]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_recipes_by_vectors(df, recipe_vectors, recipe_idx, top_n=10):

    # Вектор выбранного рецепта
    new_embedding = recipe_vectors[recipe_idx].reshape(1, -1)

    similarities = cosine_similarity(new_embedding, recipe_vectors).flatten()

    # - сам рецепт
    similarities[recipe_idx] = -1

    # индексы топ
    top_idx = similarities.argsort()[-top_n:][::-1]

    #  топ-n рецептов
    top_recipes = df.iloc[top_idx].copy()
    top_recipes['similarity'] = similarities[top_idx]  # косинусная похожесть

    print(f"Выбранный рецепт (индекс {recipe_idx}):")
    print(f"Название: {df.loc[recipe_idx, 'name']}")
    print(f"Ингредиенты: {df.loc[recipe_idx, 'ingredients_list']}\n")

    print(f"Топ-{top_n} похожих рецептов:")
    for i, row in top_recipes.iterrows():
        print(f"{i}. {row['name']} (Схожесть: {row['similarity']:.3f})")
        print(f"   Ингредиенты: {row['ingredients_list']}\n")

    return top_recipes


In [60]:
top = find_similar_recipes_by_vectors(df, recipe_vectors, recipe_idx=278, top_n=5)


Выбранный рецепт (индекс 278):
Название: Мясо на водяной бане с травами
Ингредиенты: ['вода', 'гвоздика', 'зелень', 'лук репчатый', 'мука ржаная', 'мясо', 'перец черный', 'соль']

Топ-5 похожих рецептов:
1356. Салма баранья (Схожесть: 0.953)
   Ингредиенты: ['баранина', 'вода', 'картофель', 'лист лавровый', 'лук репчатый', 'мука пшеничная', 'перец черный', 'соль']

4718. Гарнир из запаренной гречки (Схожесть: 0.953)
   Ингредиенты: ['вода', 'зелень', 'крупа гречневая', 'лук белый', 'соль', 'тыква']

3997. Рыжики в томате (Схожесть: 0.952)
   Ингредиенты: ['вода', 'грибы', 'лист лавровый', 'лук репчатый', 'масло растительное', 'морковь', 'перец черный', 'сахар', 'соль', 'томатная паста']

2119. Гороховый суп с бараниной в мультиварке (Схожесть: 0.950)
   Ингредиенты: ['баранина', 'вода', 'горох', 'зелень', 'картофель', 'лист лавровый', 'лук репчатый', 'масло растительное', 'морковь', 'сельдерей корневой', 'соль']

5437. Рис с индейкой и сушеной вишней (Схожесть: 0.949)
   Ингредиенты: [

тут видно. что самым похожим блюдом Мясо на водяной бане с травами он считает блюдо Салма баранья. Хотя в первом написано Мясо, а во втором Баранина. Также в первом 'мука ржаная' а во втором 'мука пшеничная'

In [61]:
top = find_similar_recipes_by_vectors(df, recipe_vectors, recipe_idx=31, top_n=5)


Выбранный рецепт (индекс 31):
Название: Лeгкий Рыбно-Овощной Салат
Ингредиенты: ['авокадо', 'зелень', 'капуста пекинская', 'консервы рыбные', 'креветки', 'лимон', 'масло оливковое', 'специи']

Топ-5 похожих рецептов:
3687. Салат "Праздничный" из креветок и авокадо (Схожесть: 0.948)
   Ингредиенты: ['авокадо', 'капуста пекинская', 'креветки', 'лимон', 'листья салата', 'масло оливковое', 'соль']

1183. Салат  «Русалка» (Схожесть: 0.944)
   Ингредиенты: ['горчица', 'капуста краснокочанная', 'капуста морская', 'киви', 'креветки', 'масло оливковое', 'мед', 'сок лимонный', 'специи', 'сыр твердый', 'тунец']

3183. Салат "Морское чудо" (Схожесть: 0.944)
   Ингредиенты: ['булочка', 'консервы рыбные', 'корнишоны', 'креветки', 'майонез', 'масло растительное', 'оливки черные', 'приправа', 'яйцо куриное']

1464. Салат с крабовыми палочками и кус-кусом (Схожесть: 0.940)
   Ингредиенты: ['авокадо', 'горчица', 'зелень', 'крабовые палочки', 'кус-кус', 'масло оливковое', 'огурец', 'помидор', 'помидоры ч

тут четко сработано. салат похож на салат. и при этом все с морской тематикой

In [62]:
top = find_similar_recipes_by_vectors(df, recipe_vectors, recipe_idx=1070, top_n=5)


Выбранный рецепт (индекс 1070):
Название: Торт с клубникой, ревенем и кремом
Ингредиенты: ['бисквит', 'клубника', 'крахмал', 'миндаль', 'ревень', 'рикотта', 'сахар', 'сливки']

Топ-5 похожих рецептов:
5284. Кофейно-малиновый торт (Схожесть: 0.972)
   Ингредиенты: ['ванильная эссенция', 'вода', 'джем', 'желатин', 'кофе растворимый', 'малина', 'маскарпоне', 'мука пшеничная', 'орехи лесные', 'сахар', 'сливки', 'соль', 'шоколад белый', 'шоколад темный', 'яйцо куриное']

5571. Торт "Карибское лето" (Схожесть: 0.972)
   Ингредиенты: ['ананас', 'желе', 'загуститель для сливок', 'крахмал', 'ликер', 'маскарпоне', 'миндаль', 'мука пшеничная', 'разрыхлитель теста', 'сахар', 'сливки', 'стружка кокосовая', 'яйцо куриное']

1934. Клубнично-сливочный торт-суфле (Схожесть: 0.970)
   Ингредиенты: ['белок яичный', 'ваниль', 'вода', 'желатин', 'загуститель для сливок', 'клубника', 'молоко сгущенное', 'мука пшеничная', 'сахар', 'сахарная пудра', 'сливки', 'сок лимонный', 'соль', 'экстракт']

4118. Десерт 

тут интересно. топ 1 похожесть (Схожесть: 0.972) блюдо, которое п освоему составву оказалось максмально непохожим на оригинальное блюдо. И логически тоже удачно получилось сопоставить

## Новый частнотный анализ

In [112]:
import plotly.graph_objects as go
def frequency_fun(title, top_n = 15):
    df_salad = df[df["name"].str.contains(title)]

    cur_ingredients = df_salad["ingredients_list"].explode()
    frequency_analysis = cur_ingredients.value_counts()


    top_ingredients = frequency_analysis.head(top_n)

    ingredient_names = top_ingredients.index.tolist()
    ingredient_counts = top_ingredients.values.tolist()

    fig = go.Figure(data=[
        go.Bar(
            x=ingredient_names,
            y=ingredient_counts,
            marker_color='lightseagreen',
            text=ingredient_counts,
            textposition='outside',
            hovertemplate='<b>%{x}</b><br>Частота: %{y}<extra></extra>'
        )
])

    fig.show()

In [113]:
frequency_fun('салат')

In [114]:
frequency_fun('пицца')

In [115]:
frequency_fun('йогурт')

## Проверка на непохожесть

In [116]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def find_most_different_recipes_by_vectors(df, recipe_vectors, recipe_idx, top_n=10, exclude_self=True):

    target_embedding = recipe_vectors[recipe_idx].reshape(1, -1)
    similarities = cosine_similarity(target_embedding, recipe_vectors).flatten()

    if exclude_self:
        similarities[recipe_idx] = 1

    most_different_idx = similarities.argsort()[:top_n]

    different_recipes = df.iloc[most_different_idx].copy()
    different_recipes['similarity'] = similarities[most_different_idx]
    different_recipes['distance'] = 1 - similarities[most_different_idx]  # косинусное расстояние

    print(f"Исходный рецепт (индекс {recipe_idx}):")
    print(f"Название: {df.loc[recipe_idx, 'name']}")
    print(f"Ингредиенты: {df.loc[recipe_idx, 'ingredients_list']}")
    print("-" * 50)

    print(f"\nТоп-{top_n} самых НЕпохожих рецептов:")
    print(f"(отсортировано по возрастанию сходства)")
    print("=" * 80)

    for i, (idx, row) in enumerate(different_recipes.iterrows(), 1):
        print(f"{i}. {row['name']}")
        print(f"   Сходство: {row['similarity']:.3f} | Расстояние: {row['distance']:.3f}")
        print(f"   Ингредиенты: {row['ingredients_list']}")
        print("-" * 40)

    return different_recipes

In [117]:
top = find_most_different_recipes_by_vectors(df, recipe_vectors, recipe_idx=1070, top_n=5)


Исходный рецепт (индекс 1070):
Название: Торт с клубникой, ревенем и кремом
Ингредиенты: ['бисквит', 'клубника', 'крахмал', 'миндаль', 'ревень', 'рикотта', 'сахар', 'сливки']
--------------------------------------------------

Топ-5 самых НЕпохожих рецептов:
(отсортировано по возрастанию сходства)
1. Рагу
   Сходство: 0.238 | Расстояние: 0.762
   Ингредиенты: ['лук репчатый', 'морковь', 'овощи', 'огурец', 'перец болгарский', 'помидор']
----------------------------------------
2. Салат "Деревенский"
   Сходство: 0.241 | Расстояние: 0.759
   Ингредиенты: ['майонез', 'морковь', 'огурец соленый', 'сухарики']
----------------------------------------
3. Салат из куриных сердечек с луком
   Сходство: 0.244 | Расстояние: 0.756
   Ингредиенты: ['лук репчатый', 'майонез', 'сердечки куриные']
----------------------------------------
4. Цветы из острого перца
   Сходство: 0.247 | Расстояние: 0.753
   Ингредиенты: ['лук-порей', 'перец красный жгучий']
----------------------------------------
5. Ово

In [119]:
sentences = []

for ingredients, title in zip(recipes, recipe_names):
    title_words = title.lower().split()
    sentences.append(ingredients + title_words)

model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    epochs=20
)


In [125]:
import numpy as np

def recipe_embedding(recipe, title, model, alpha=0.2):
    ing_vecs = [model.wv[w] for w in recipe if w in model.wv]
    title_vecs = [model.wv[w] for w in title.lower().split() if w in model.wv]

    ing_vec = np.mean(ing_vecs, axis=0) if ing_vecs else np.zeros(model.vector_size)
    title_vec = np.mean(title_vecs, axis=0) if title_vecs else np.zeros(model.vector_size)

    return alpha * title_vec + (1 - alpha) * ing_vec


recipe_vectors = np.array([
    recipe_embedding(r, t, model)
    for r, t in zip(recipes, recipe_names)
])


In [126]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
recipe_3d = pca.fit_transform(recipe_vectors)

In [127]:
salad_vec = model.wv['салат']
salad_3d = pca.transform([salad_vec])[0]

In [128]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    recipe_vectors,
    salad_vec.reshape(1, -1)
).flatten()


In [129]:
import plotly.express as px

fig = px.scatter_3d(
    x=recipe_3d[:, 0],
    y=recipe_3d[:, 1],
    z=recipe_3d[:, 2],
    color=similarities,
    color_continuous_scale='RdYlGn_r',
    hover_name=recipe_names,
    title="Пространство рецептов: близость к слову 'салат'"
)

# точка "салат"
fig.add_scatter3d(
    x=[salad_3d[0]],
    y=[salad_3d[1]],
    z=[salad_3d[2]],
    mode='markers+text',
    marker=dict(size=8, color='black'),
    text=['салат'],
    textposition='top center',
    name='салат'
)

fig.show()
